# Calcul du taux de décès sur 2 ans

Calcule `(décès t + décès t+1) / pop t` pour chaque pays et année.


In [ ]:
import pandas as pd
import numpy as np
import os

# Chargement
total_deces = pd.read_excel('data/total_deces.xlsx', header=0)
pop_totale  = pd.read_excel('data/Pop_tot_fr.xlsx', header=0)


In [ ]:
# Conversion en format long
deces_long = total_deces.rename(columns={'TIME': 'Pays'}).melt(
    id_vars='Pays', var_name='Annee', value_name='Deces'
)
deces_long['Annee'] = deces_long['Annee'].astype(int)
deces_long['Deces'] = pd.to_numeric(
    deces_long['Deces'].astype(str).str.replace(':', 'nan').str.strip(), errors='coerce'
)

pop_long = pop_totale.rename(columns={'TIME': 'Pays'}).melt(
    id_vars='Pays', var_name='Annee', value_name='Pop_totale'
)
pop_long['Annee'] = pop_long['Annee'].astype(int)
pop_long['Pop_totale'] = pd.to_numeric(
    pop_long['Pop_totale'].astype(str).str.replace(':', 'nan').str.strip(), errors='coerce'
)


In [ ]:
# Calcul du taux : (décès t + décès t+1) / pop t × 100
ANNEE_DEBUT, ANNEE_FIN = 2004, 2023
all_results = []

for pays in deces_long['Pays'].unique():
    d = deces_long[(deces_long['Pays'] == pays) & (deces_long['Annee'].between(ANNEE_DEBUT, ANNEE_FIN + 1))]
    p = pop_long[(pop_long['Pays'] == pays) & (pop_long['Annee'].between(ANNEE_DEBUT, ANNEE_FIN))]

    D_t  = d.set_index('Annee')['Deces']
    D_t1 = D_t.shift(-1)

    df_r = pd.DataFrame({'D_t': D_t, 'D_t1': D_t1,
                         'Pop_totale': p.set_index('Annee')['Pop_totale']}).loc[ANNEE_DEBUT:ANNEE_FIN]
    df_r['Taux_deces_2ans'] = (df_r['D_t'] + df_r['D_t1']) / df_r['Pop_totale'] * 100
    df_r['Pays'] = pays
    df_r.index.name = 'Annee'
    all_results.append(df_r.reset_index())

df_all = pd.concat(all_results, ignore_index=True)
valid  = df_all.dropna(subset=['Taux_deces_2ans'])
print(f'{valid.Pays.nunique()} pays x {valid.Annee.nunique()} annees ({valid.Annee.min()}-{valid.Annee.max()})')


In [ ]:
# Export
df_pivot = valid.pivot(index='Pays', columns='Annee', values='Taux_deces_2ans').round(3)
df_pivot.columns.name = None
df_pivot.index.name   = 'Pays'

os.makedirs('data', exist_ok=True)
df_pivot.to_excel('data/Taux_Deces_2ans.xlsx')
print('Taux_Deces_2ans.xlsx exporte.')
